In [ ]:
import polars as pl
import json
import os

def generate_llm_data_profile_stream(file_path, batch_size=100_000):
    print(f"Starting profile generation for: {file_path}")
    
    # 1. Preliminary Scan: Get total rows and schema efficiently
    try:
        lf = pl.scan_csv(file_path)
        total_rows = lf.select(pl.len()).collect().item()
    except Exception as e:
        print(f"Error reading file metadata: {e}")
        return

    # Initialize profile structure
    profile = {
        "dataset_metadata": {
            "filename": file_path,
            "total_records": total_rows
        },
        "columns": {}
    }

    # Tracking structures for incremental aggregation
    col_trackers = {}

    # 2. Batched Processing
    reader = pl.read_csv_batched(file_path, batch_size=batch_size)
    
    batch_idx = 0
    while True:
        batches = reader.next_batches(1) 
        if not batches:
            break
        
        df_batch = batches[0]
        batch_idx += 1
        
        for col in df_batch.columns:
            if col not in col_trackers:
                col_trackers[col] = {
                    "total_non_null": 0,
                    "is_json": False,
                    "checked_type": False,
                    "json_key_counts": {}, 
                    "json_total_entries": 0,
                    "max_key_len": -1,
                    "best_example": None,
                    "scalar_example": None,
                    "unique_values": set(),          # Outer unique values
                    "nested_unique_values": {}       # Inner JSON unique values
                }
            
            tracker = col_trackers[col]
            series = df_batch[col].drop_nulls()
            
            if series.is_empty():
                continue

            tracker["total_non_null"] += len(series)

            # Determine column type using the first non-null value encountered
            if not tracker["checked_type"]:
                first_val = str(series[0]).strip()
                tracker["scalar_example"] = first_val
                if first_val.startswith(("{", "[")):
                    tracker["is_json"] = True
                tracker["checked_type"] = True

            # Process Outer Unique Values for non-JSON columns
            if not tracker["is_json"]:
                tracker["unique_values"].update(series.cast(pl.Utf8).to_list())

            # If it's a JSON column, process row-by-row
            if tracker["is_json"]:
                for val in series.to_list():
                    try:
                        data = json.loads(val)
                        entries = data if isinstance(data, list) else [data]
                        
                        for entry in entries:
                            if not isinstance(entry, dict): continue
                            
                            tracker["json_total_entries"] += 1
                            current_keys = [k for k, v in entry.items() if v is not None and str(v).strip() != ""]
                            
                            for k, v in entry.items():
                                # Update global key counts
                                increment = 1 if k in current_keys else 0
                                tracker["json_key_counts"][k] = tracker["json_key_counts"].get(k, 0) + increment

                                # Track unique values for the nested key
                                if k not in tracker["nested_unique_values"]:
                                    tracker["nested_unique_values"][k] = set()
                                
                                if v is not None:
                                    tracker["nested_unique_values"][k].add(str(v))

                            # Track best example (most populated fields)
                            if len(current_keys) > tracker["max_key_len"]:
                                tracker["max_key_len"] = len(current_keys)
                                tracker["best_example"] = entry
                    except:
                        continue

    # 3. Finalize Aggregation
    print("Finalizing profile...")
    
    for col, tracker in col_trackers.items():
        presence_pct = (tracker["total_non_null"] / total_rows) * 100 if total_rows > 0 else 0
        
        if tracker["total_non_null"] == 0:
             profile["columns"][col] = {"status": "all_null", "presence_pct": 0}
             continue

        if tracker["is_json"]:
            field_analysis = {}
            total_json_entries = tracker["json_total_entries"]
            
            if total_json_entries > 0:
                for key, count in tracker["json_key_counts"].items():
                    nested_vals = list(tracker["nested_unique_values"].get(key, []))
                    nested_unique_count = len(nested_vals)
                    
                    field_info = {
                        "presence_pct": round((count / total_json_entries) * 100, 5),
                        "is_mandatory": (count / total_json_entries) == 1.0,
                        "unique_count": nested_unique_count
                    }
                    
                    if nested_unique_count < 100:
                        field_info["unique_values_list"] = nested_vals
                        
                    field_analysis[key] = field_info
            
            profile["columns"][col] = {
                "type": "json_structure", 
                "column_presence_pct": round(presence_pct, 2),
                "nested_schema": field_analysis,
                "representative_example": tracker["best_example"]
            }
        else:
            unique_vals = list(tracker["unique_values"])
            unique_count = len(unique_vals)
            
            col_data = {
                "type": "scalar_text",
                "column_presence_pct": round(presence_pct, 2),
                "unique_count": unique_count,
                "example_value": tracker["scalar_example"]
            }
            
            if unique_count < 100:
                col_data["unique_values_list"] = unique_vals
                
            profile["columns"][col] = col_data

    # Output to File
    output_dir = "../Data"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    output_path = os.path.join(output_dir, "filter_structure.json")
    with open(output_path, "w") as f:
        json.dump(profile, f, indent=2)
    
    print(f"Profile saved to: {output_path}")

# Run
if __name__ == "__main__":
    generate_llm_data_profile_stream("../Data/fresh-grad-filter.csv")

In [ ]:
import json

# ── Load data ─────────────────────────────────────────────────────────────────
with open("../Data/filter_structure.json", "r", encoding="utf-8") as f:
    data = json.load(f)

total_records = data["dataset_metadata"]["total_records"]
columns       = data["columns"]

# ── Utilities ─────────────────────────────────────────────────────────────────
def esc(s):
    return (str(s)
            .replace("&", "&amp;")
            .replace("<", "&lt;")
            .replace(">", "&gt;")
            .replace('"', "&quot;"))

# ── HTML fragment builders ────────────────────────────────────────────────────
def presence_badge(pct):
    if pct == 100.0:   cls, label = "badge-success", "complete"
    elif pct >= 80:    cls, label = "badge-info",    "high"
    elif pct >= 50:    cls, label = "badge-warning",  "partial"
    else:              cls, label = "badge-danger",   "low"
    return (
        f'<span class="badge {cls}" '
        f'aria-label="Presence: {pct}% — {label}">{pct}%</span>'
    )

def render_example(value):
    if value is None:
        return '<span class="muted" aria-label="No example">—</span>'
    if isinstance(value, (dict, list)):
        code = esc(json.dumps(value, indent=2, ensure_ascii=False))
        return f'<pre class="code-light" aria-label="Example JSON value">{code}</pre>'
    return f'<code class="inline-code">{esc(str(value))}</code>'

def render_tag_list(values, size=""):
    if not values:
        return None
    cls   = f"tag{' tag-sm' if size == 'sm' else ''}"
    items = "".join(
        f'<li><span class="{cls}">{esc(str(v))}</span></li>'
        for v in values
    )
    return (
        f'<ul class="tag-list" '
        f'aria-label="{len(values)} unique value{"s" if len(values) != 1 else ""}">'
        f'{items}</ul>'
    )

def render_nested_table(schema, col_name):
    if not schema:
        return '<p class="muted">No nested schema defined.</p>'
    rows = ""
    for i, (field, props) in enumerate(schema.items()):
        is_mandatory = props.get("is_mandatory", False)
        presence     = props.get("presence_pct", 0)
        uc           = props.get("unique_count")
        uv           = props.get("unique_values_list") or []
        kind         = "mandatory" if is_mandatory else "optional"
        row_cls      = "row-mandatory" if is_mandatory else ("row-even" if i % 2 == 0 else "row-odd")
        pill = (
            ' <span class="pill-mandatory" aria-hidden="true">mandatory</span>'
            if is_mandatory else ""
        )
        uv_html = render_tag_list(uv, size="sm") or '<span class="muted" aria-label="No unique values">—</span>'
        uc_html = f"{uc:,}" if uc else '<span class="muted" aria-label="Not available">—</span>'
        rows += f"""
              <tr class="{row_cls}">
                <th scope="row">{esc(field)}{pill}<span class="sr-only"> ({kind})</span></th>
                <td class="center">{presence_badge(presence)}</td>
                <td class="center">{uc_html}</td>
                <td>{uv_html}</td>
              </tr>"""
    return f"""
          <div class="table-wrap">
            <table aria-label="Nested schema for column {esc(col_name)}">
              <caption class="sr-only">Nested schema fields for column {esc(col_name)}</caption>
              <thead>
                <tr>
                  <th scope="col">Field</th>
                  <th scope="col" class="center">Presence</th>
                  <th scope="col" class="center">Unique count</th>
                  <th scope="col">Unique values</th>
                </tr>
              </thead>
              <tbody>{rows}
              </tbody>
            </table>
          </div>"""

def render_col_section(col_name, col_info, idx):
    col_type  = col_info.get("type", "unknown")
    uc        = col_info.get("unique_count", 0)
    type_pill = f'<span class="pill-type" aria-label="Type: {esc(col_type)}">{esc(col_type)}</span>'

    metrics = f"""
        <dl class="metrics" aria-label="Summary metrics for {esc(col_name)}">
          <div class="metric">
            <dt>Column presence</dt>
            <dd>{presence_badge(col_info.get('column_presence_pct', 0))}</dd>
          </div>
          <div class="metric">
            <dt>Unique count</dt>
            <dd><span class="badge badge-neutral" aria-label="{uc:,} unique values">{uc:,}</span></dd>
          </div>
          <div class="metric">
            <dt>Example value</dt>
            <dd>{render_example(col_info.get('example_value'))}</dd>
          </div>
        </dl>"""

    extras = ""

    uv      = col_info.get("unique_values_list")
    uv_html = render_tag_list(uv) if uv else None
    if uv_html:
        extras += f"""
        <section aria-label="Unique values for column {esc(col_name)}">
          <h3>Unique values</h3>
          {uv_html}
        </section>"""

    if col_type == "json_structure":
        extras += f"""
        <section aria-label="Nested schema for column {esc(col_name)}">
          <h3>Nested schema</h3>
          {render_nested_table(col_info.get('nested_schema', {}), col_name)}
        </section>"""

    rep = col_info.get("representative_example")
    if rep:
        code = esc(json.dumps(rep, indent=2, ensure_ascii=False))
        extras += f"""
        <section aria-label="Representative example for column {esc(col_name)}">
          <h3>Representative example</h3>
          <pre class="code-dark" aria-label="Representative JSON example">{code}</pre>
        </section>"""

    return f"""
      <article class="col-card" aria-labelledby="col-{idx}">
        <header class="col-header">
          <h2 id="col-{idx}">{esc(col_name)}</h2>
          {type_pill}
        </header>
        {metrics}
        {extras}
      </article>"""

# ── Build cards ───────────────────────────────────────────────────────────────
cards_html = "\n".join(
    render_col_section(name, info, i)
    for i, (name, info) in enumerate(columns.items())
)

# ── CSS (plain string — keeps curly braces out of the f-string) ───────────────
CSS = """
  *, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }

  body {
    font-family: 'DM Sans', system-ui, sans-serif;
    background: #fafaf8; color: #1a1a18;
    line-height: 1.6; font-size: 16px;
  }

  .skip-link {
    position: absolute; top: -48px; left: 16px;
    background: #534ab7; color: #fff;
    padding: 8px 18px; border-radius: 0 0 8px 8px;
    font-weight: 700; font-size: 0.875em;
    text-decoration: none; z-index: 9999;
    transition: top 0.15s ease;
  }
  .skip-link:focus { top: 0; }

  .sr-only {
    position: absolute; width: 1px; height: 1px;
    padding: 0; margin: -1px; overflow: hidden;
    clip: rect(0,0,0,0); white-space: nowrap; border: 0;
  }

  .container { max-width: 80%; margin: 0 auto; padding: 0 24px; }

  .page-header {
    padding: 52px 0 36px;
    border-bottom: 0.5px solid #d3d1c7;
    margin-bottom: 36px;
  }
  h1 {
    font-size: 2em; font-weight: 500; color: #1a1a18;
    margin-bottom: 10px; letter-spacing: -0.01em;
  }
  .subtitle { color: #888780; font-size: 0.95em; }
  .subtitle span + span::before { content: " · "; color: #d3d1c7; padding: 0 2px; }

  .col-card {
    background: #fff; border: 0.5px solid #d3d1c7;
    border-radius: 12px; padding: 24px 28px; margin-bottom: 24px;
  }
  .col-header {
    display: flex; align-items: center; gap: 12px;
    padding-bottom: 18px; border-bottom: 0.5px solid #ececea;
    margin-bottom: 18px;
  }
  h2 { font-size: 1.15em; font-weight: 600; color: #1a1a18; letter-spacing: -0.01em; }
  h3 {
    font-size: 0.7em; font-weight: 700; color: #888780;
    text-transform: uppercase; letter-spacing: 0.09em; margin: 24px 0 10px;
  }

  .metrics {
    display: flex; flex-wrap: wrap;
    background: #f9f9f7; border: 0.5px solid #d3d1c7;
    border-radius: 8px; overflow: hidden;
  }
  .metric { padding: 14px 22px; border-right: 0.5px solid #d3d1c7; min-width: 160px; }
  .metric:last-child { border-right: none; }
  .metric dt {
    font-size: 0.68em; font-weight: 700; color: #888780;
    text-transform: uppercase; letter-spacing: 0.08em; margin-bottom: 7px;
  }
  .metric dd { margin: 0; }

  .badge { display: inline-block; padding: 3px 12px; border-radius: 20px; font-size: 0.78em; font-weight: 700; line-height: 1.4; }
  .badge-success { background: #eaf3de; color: #3b6d11; }
  .badge-info    { background: #e6f1fb; color: #185fa5; }
  .badge-warning { background: #faeeda; color: #854f0b; }
  .badge-danger  { background: #fcebeb; color: #a32d2d; }
  .badge-neutral { background: #f1efe8; color: #5f5e5a; }

  .pill-type {
    display: inline-block; background: #eeedfe; color: #534ab7;
    font-size: 0.72em; padding: 3px 12px; border-radius: 20px; font-weight: 700;
  }
  .pill-mandatory {
    display: inline-block; background: #534ab7; color: #eeedfe;
    font-size: 0.68em; padding: 1px 8px; border-radius: 20px;
    font-weight: 700; vertical-align: middle; margin-left: 8px;
  }

  .tag-list { list-style: none; padding: 0; margin: 0; display: flex; flex-wrap: wrap; gap: 5px; }
  .tag, .tag-sm {
    display: inline-block; background: #e6f1fb; color: #185fa5;
    border: 0.5px solid #b5d4f4; padding: 3px 10px; border-radius: 4px; font-size: 0.78em;
  }
  .tag-sm { font-size: 0.72em; padding: 2px 8px; }

  .table-wrap { overflow-x: auto; border-radius: 10px; border: 0.5px solid #d3d1c7; }
  table { width: 100%; border-collapse: collapse; font-size: 0.84em; }
  thead tr { background: #2c2c2a; }
  thead th { padding: 10px 16px; color: #f1efe8; font-weight: 500; text-align: left; white-space: nowrap; }
  tbody th { padding: 9px 16px; font-weight: 600; text-align: left; vertical-align: top; }
  tbody td { padding: 9px 16px; vertical-align: top; }
  .row-mandatory { background: #eeedfe; }
  .row-mandatory th, .row-mandatory td { color: #3c3489; border-bottom: 0.5px solid #afa9ec; }
  .row-even { background: #fff; }
  .row-even th, .row-even td { border-bottom: 0.5px solid #e8e7e4; color: #1a1a18; }
  .row-odd  { background: #f9f9f8; }
  .row-odd th, .row-odd td { border-bottom: 0.5px solid #e8e7e4; color: #1a1a18; }
  .center { text-align: center; }

  code, pre { font-family: 'DM Mono', 'Fira Code', ui-monospace, monospace; }
  .inline-code { background: #f1efe8; color: #2c2c2a; padding: 2px 8px; border-radius: 4px; font-size: 0.85em; }
  .code-light { background: #f1efe8; color: #2c2c2a; padding: 10px 14px; border-radius: 8px; font-size: 0.78em; overflow-x: auto; line-height: 1.5; margin: 0; }
  .code-dark  { background: #2c2c2a; color: #d3d1c7; padding: 18px 22px; border-radius: 10px; font-size: 0.8em; overflow-x: auto; line-height: 1.65; margin: 0; }

  .muted { color: #888780; font-style: italic; }

  .page-footer {
    text-align: center; color: #b4b2a9; font-size: 0.78em;
    padding: 36px 0 28px; border-top: 0.5px solid #d3d1c7; margin-top: 8px;
  }
"""

# ── Assemble & write ──────────────────────────────────────────────────────────
html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Dataset Structure Overview</title>
  <link rel="preconnect" href="https://fonts.googleapis.com">
  <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
  <link href="https://fonts.googleapis.com/css2?family=DM+Mono:wght@400;500&family=DM+Sans:wght@400;500;600;700&display=swap" rel="stylesheet">
  <style>{CSS}</style>
</head>
<body>

  <a class="skip-link" href="#main">Skip to main content</a>

  <div class="container">
    <header class="page-header">
      <h1>Dataset structure overview</h1>
      <p class="subtitle">
        <span>{total_records:,} records</span>
        <span>{len(columns)} columns</span>
      </p>
    </header>

    <main id="main">
      {cards_html}
    </main>

    <footer class="page-footer">
      Data source: final_structure.json
    </footer>
  </div>

</body>
</html>"""

output_path = "../Data/dataset_structure.html"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(html)

print(f"✅ Saved to {output_path}")